# עבודת סוכות: יישום RAG וניתוח מאמר ב-MongoDB Atlas


**המאמר:** Jaju R, Varshney S, Gupta P, Bihani P, Karim HMR.
*Inventory Control Mechanism of the Pharmacy Store of a Recently Established
National Institute in Eastern India: A Cross-Sectional, Investigative Analysis.*


## 1. חיבור ל-MongoDB Atlas

In [23]:
from pymongo import MongoClient
from pymongo.operations import SearchIndexModel
from pymongo.errors import OperationFailure
from pprint import pprint
from dotenv import load_dotenv
import os, time

load_dotenv()
print("MONGO_URI loaded:", bool(os.getenv("MONGO_URI")))

MONGO_URI loaded: True


In [24]:
client = MongoClient(os.getenv("MONGO_URI"))

try:
    client.admin.command("ping")
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(f"Connection failed: {e}")

Pinged your deployment. You successfully connected to MongoDB!


## 2. בסיס הנתונים הקיים

In [25]:
db = client["pharmacy_nosql"]

products = db["products"]          
ARTICLE_COLLECTION = "article_chunks"   

print("collections before:", db.list_collection_names())
print("products documents:", products.count_documents({}))

collections before: ['article_chunks', 'products']
products documents: 24


## 3. Ingestion — טעינת המאמר מתוך קובץ PDF

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("data/pharmacy_inventory_article.pdf")
pages = loader.load()
print(f"Total Pages: {len(pages)}")

cleaned_pages = [p for p in pages if len(p.page_content.split()) > 20]
print(f"Total Cleaned Pages: {len(cleaned_pages)}")

Total Pages: 10
Total Cleaned Pages: 10


### חלוקה לקטעים (Chunks)

In [27]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=750,
    chunk_overlap=150
)
docs_chunks = text_splitter.split_documents(cleaned_pages)

print(f"Total Chunks: {len(docs_chunks)}")
print(docs_chunks[0].page_content[:300])

Total Chunks: 56
Review began
 11/14/2023 
Review ended
 11/19/2023 
Published
 11/29/2023
© Copyright 
2023
Jaju et al. This is an open access article
distributed under the terms of the Creative
Commons Attribution License CC-BY 4.0.,
which permits unrestricted use, distribution,
and reproduction in any medium, pro


## 4. יצירת הווקטורים (Embeddings)

כל chunk מקבל embedding **משלו** — מהטקסט שלו.

In [ ]:
import voyageai

voyage = voyageai.Client(api_key=os.getenv("VOYAGEAI_API_KEY"))

EMBED_MODEL = "voyage-4"
EMBED_DIMS = len(voyage.embed(["dimension probe"], model=EMBED_MODEL,
                              input_type="document").embeddings[0])
print("embedding dimensions:", EMBED_DIMS)

embedding dimensions: 1024


In [29]:
assert ARTICLE_COLLECTION not in ("products",), "אסור לגעת בקולקשנים הקיימים"

chunks_collection = db[ARTICLE_COLLECTION]
chunks_collection.delete_many({})   
print(f"{ARTICLE_COLLECTION}: {chunks_collection.count_documents({})} documents")

article_chunks: 0 documents


In [30]:
BATCH_SIZE = 64
records = []

for start in range(0, len(docs_chunks), BATCH_SIZE):
    batch = docs_chunks[start:start + BATCH_SIZE]
    texts = [chunk.page_content for chunk in batch]

    embeddings = voyage.embed(
        texts,
        model=EMBED_MODEL,
        input_type="document"
    ).embeddings

    for chunk, embedding in zip(batch, embeddings):
        records.append({
            "text": chunk.page_content,
            "metadata": chunk.metadata,
            "embedding": embedding,
        })

    print(f"Embedded {len(records)} / {len(docs_chunks)} chunks")

chunks_collection.insert_many(records)
print(f"Inserted {chunks_collection.count_documents({})} documents "
      f"into {ARTICLE_COLLECTION}")

Embedded 56 / 56 chunks
Inserted 56 documents into article_chunks


## 5. Vector Search Index

חשוב: יוצרים את האינדקס **אחרי** הכנסת המסמכים — אי אפשר לאנדקס קולקשן שעדיין לא קיים.

In [31]:
INDEX_NAME = "vector_search_index"

definition = {
    "fields": [
        {
            "type": "vector",
            "path": "embedding",
            "numDimensions": EMBED_DIMS,
            "similarity": "cosine"
        }
    ]
}

search_index_model = SearchIndexModel(
    definition=definition,
    name=INDEX_NAME,
    type="vectorSearch"
)

try:
    chunks_collection.create_search_index(search_index_model)
    print(f"Created search index {INDEX_NAME!r}")
except OperationFailure as e:
    print(f"Index not created, continuing with the existing one: {e}")

Created search index 'vector_search_index'


In [ ]:
for attempt in range(60):
    info = next((i for i in chunks_collection.list_search_indexes()
                 if i["name"] == INDEX_NAME), None)
    if info and info.get("queryable"):
        print(f"Index {INDEX_NAME!r} is queryable (status={info.get('status')})")
        break
    print(f"  waiting... status={info.get('status') if info else 'missing'}")
    time.sleep(5)
else:
    print("Index is still not queryable - the search below will return nothing")

Index 'vector_search_index' is queryable (status=READY)


## 6. Retrieval — חיפוש וקטורי

In [ ]:
def retrieve(query_text, limit=5):
    query_vector = voyage.embed(
        [query_text],
        model=EMBED_MODEL,
        input_type="query"
    ).embeddings[0]

    pipeline = [
        {
            "$vectorSearch": {
                "exact": True,
                "index": INDEX_NAME,
                "limit": limit,
                "path": "embedding",
                "queryVector": query_vector,
            }
        },
        {
            "$project": {
                "_id": 0,
                "text": 1,
                "page": "$metadata.page",
                "score": {"$meta": "vectorSearchScore"}
            }
        }
    ]
    return list(chunks_collection.aggregate(pipeline))

In [34]:
query_text = "Which drugs require the most stringent inventory control and why?"
results = retrieve(query_text)

print(f"{len(results)} results\n")
for r in results:
    print(f"page {r['page']}  score {r['score']:.4f}")
    print(f"  {r['text'][:200].strip()}...\n")

0 results



## 7. RAG — תשובה מבוססת מקור

In [35]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.1,
    api_key=os.getenv("GOOGLE_API_KEY")
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question based ONLY on the following context:\n\n{context}"),
    ("human", "{query}")
])

rag_chain = prompt | llm | StrOutputParser()

In [ ]:
def ask(question, limit=5):
    hits = retrieve(question, limit=limit)
    context = "\n\n".join(h["text"] for h in hits)
    answer = rag_chain.invoke({"context": context, "query": question})
    pages = sorted({h["page"] for h in hits})
    return answer, pages


answer, pages = ask(query_text)
print(answer)   
print(f"\n--- based on pages: {pages}")

C:\Users\yarde\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided text, the drugs requiring the most stringent inventory control are:

1. **Category A Drugs:** These require strict managerial control, accurate data-driven demand prediction, budgetary scrutiny, minimum safety stock, staggered purchase orders, frequent stock-taking, and rational management policies.
2. **Category I Drugs:** These require strict managerial control—as well as strict control over their prescription and utilization—because they are **either expensive or vital** (crucial for patient care) and must always be kept in stock. Because of their higher cost, holding a low buffer stock under strict control prevents the locking up of capital. 
3. **Vital (V) Drugs:** These require continuous availability and appropriate safety stock with **zero tolerance for stockout options** due to their critical nature.

--- based on pages: [5, 9]


In [37]:
for q in [
    "Which drugs require the most stringent inventory control and why?",
    "What problems cause drug stock-outs in a hospital pharmacy store?",
]:
    answer, pages = ask(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print(f"   pages: {pages}\n{'-' * 70}\n")

C:\Users\yarde\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Q: Which drugs require the most stringent inventory control and why?
A: Based on the provided context, the drugs that require the most stringent inventory control are **Category A drugs**, **Category I drugs** (from the combined ABC and VED analysis), and **Vital (V) drugs**. 

They require the most strict control for the following reasons:

* **They are expensive or vital:** Category I drugs are either expensive or vital, making them crucial for patient care.
* **To prevent locking up capital:** Because of their higher cost, strict control and holding a low buffer stock are necessary to avoid locking up capital.
* **Critical availability / Zero tolerance for stockouts:** They are crucial and must always be maintained in stock; vital medicines specifically require continuous availability and appropriate safety stock with **zero tolerance for stockouts**.
* **Budgetary and demand management:** Category A drugs require strict control for accurate data-driven demand prediction, scrutiny o

C:\Users\yarde\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Q: What problems cause drug stock-outs in a hospital pharmacy store?
A: Based on the provided text, the causes and delays leading to drug stockouts and shortages are mainly related to:

* **Rural location of the institute** 
* **Vendor management** 
* **Communication hurdles**
   pages: [0, 1, 6]
----------------------------------------------------------------------

